# Cross Attention Fusion Experiment

This notebook implements a comparison experiment using 3D DenseNet for feature extraction and Cross Attention for multi-modal fusion (MRI + PET).

In [6]:
import sys
import os
import json
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from monai.data import Dataset as MonaiDataset

# Add project root to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from datasets.ADNI import ADNI, ADNI_transform
from utils.metrics import calculate_metrics
from Comparison_Experiment.model.densenet3d import densenet121_3d
from Comparison_Experiment.model.cross_attention import CrossModalFusion
from Comparison_Experiment.model.classifier import Classifier

In [7]:
# -------------------- Configuration --------------------
def load_cfg(path):
    with open(path) as f: 
        return json.load(f)

class Cfg:
    def __init__(self, d):
        self.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        for k, v in d.items(): 
            setattr(self, k, v)

config_path = "config/config.json"
cfg = Cfg(load_cfg(config_path))

# Override for this experiment
# cfg.checkpoint_dir is already set to 'checkpoints' in the local config, which will create 'Comparison_Experiment/checkpoints'
os.makedirs(cfg.checkpoint_dir, exist_ok=True)

print(f"Device: {cfg.device}")
print(f"Batch Size: {cfg.batch_size}")
print(f"Epochs: {cfg.num_epochs}")

Device: cuda:0
Batch Size: 2
Epochs: 20


In [8]:
# -------------------- Model Definition --------------------
class CrossAttnNet(nn.Module):
    def __init__(self, num_classes=2, drop_rate=0.2):
        super(CrossAttnNet, self).__init__()
        # 1. Encoders (Shared weights or separate? Usually separate for multi-modal)
        self.mri_encoder = densenet121_3d(in_channels=1, drop_rate=drop_rate)
        self.pet_encoder = densenet121_3d(in_channels=1, drop_rate=drop_rate)
        
        # DenseNet121 output features is 1024
        feature_dim = 1024 
        embed_dim = 256
        
        # 2. Cross Attention Fusion
        self.fusion = CrossModalFusion(in_dim1=feature_dim, in_dim2=feature_dim, embed_dim=embed_dim, dropout=drop_rate)
        
        # 3. Classifier
        # Fusion output is concat of two attended features: 2 * embed_dim
        classifier_in_dim = 2 * embed_dim
        self.classifier = Classifier(in_dim=classifier_in_dim, num_classes=num_classes, p_drop=drop_rate)

    def forward(self, mri, pet):
        # Extract features
        feat_mri = self.mri_encoder(mri) # (B, 1024)
        feat_pet = self.pet_encoder(pet) # (B, 1024)
        
        # Cross Attention Fusion
        fused = self.fusion(feat_mri, feat_pet) # (B, 512)
        
        # Classification
        logits = self.classifier(fused) # (B, 2)
        return logits

In [9]:
# -------------------- Training Helper Functions --------------------
def train_epoch(model, loader, optimizer, scaler, criterion, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for batch in loader:
        mri = batch["MRI"].to(device)
        pet = batch["PET"].to(device)
        label = batch["label"].to(device)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            logits = model(mri, pet)
            loss = criterion(logits, label)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * mri.size(0)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(label.cpu().numpy())
        
    epoch_loss = running_loss / len(loader.dataset)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return epoch_loss, acc

def val_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in loader:
            mri = batch["MRI"].to(device)
            pet = batch["PET"].to(device)
            label = batch["label"].to(device)
            
            with autocast():
                logits = model(mri, pet)
                loss = criterion(logits, label)
            
            running_loss += loss.item() * mri.size(0)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    epoch_loss = running_loss / len(loader.dataset)
    metrics = calculate_metrics(all_labels, all_preds, all_probs)
    return epoch_loss, metrics

In [10]:
# -------------------- Main Execution Loop --------------------

# 1. Load Data
full_dataset = ADNI(cfg.label_file, cfg.mri_dir, cfg.pet_dir, cfg.task, cfg.augment)
full_ds = full_dataset.data_dict
labels = [d["label"] for d in full_ds]

skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)

# 2. Cross Validation
for fold, (train_idx, test_idx) in enumerate(skf.split(full_ds, labels), start=1):
    print(f"\n{'='*20} Fold {fold} {'='*20}")
    
    # Split Train/Val/Test
    train_subset = [full_ds[i] for i in train_idx]
    test_subset = [full_ds[i] for i in test_idx]
    
    # Inner split for validation (10% of training set)
    train_idx_inner, val_idx_inner = train_test_split(
        np.arange(len(train_idx)), test_size=0.1, stratify=[labels[i] for i in train_idx], random_state=cfg.seed
    )
    
    train_data = [train_subset[i] for i in train_idx_inner]
    val_data = [train_subset[i] for i in val_idx_inner]
    
    # Create DataLoaders
    train_tfm, _ = ADNI_transform(augment=cfg.augment)
    val_tfm, _ = ADNI_transform(augment=False)
    
    train_ds = MonaiDataset(data=train_data, transform=train_tfm)
    val_ds = MonaiDataset(data=val_data, transform=val_tfm)
    test_ds = MonaiDataset(data=test_subset, transform=val_tfm)
    
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=True)
    
    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
    
    # Initialize Model
    model = CrossAttnNet(num_classes=cfg.nb_class, drop_rate=cfg.dropout_rate).to(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.num_epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler()
    
    best_val_acc = 0.0
    best_model_path = os.path.join(cfg.checkpoint_dir, f"fold{fold}_best.pth")
    
    # Training Loop
    for epoch in range(cfg.num_epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, scaler, criterion, cfg.device)
        scheduler.step()
        val_loss, val_metrics = val_epoch(model, val_loader, criterion, cfg.device)
        
        print(f"Epoch {epoch+1}/{cfg.num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_metrics['ACC']:.4f} AUC: {val_metrics['AUC']:.4f}")
        
        if val_metrics['ACC'] > best_val_acc:
            best_val_acc = val_metrics['ACC']
            torch.save(model.state_dict(), best_model_path)
            print(f"  >>> New Best Model Saved (Acc: {best_val_acc:.4f})")
            
    # Final Test
    print(f"Testing Fold {fold}...")
    model.load_state_dict(torch.load(best_model_path))
    test_loss, test_metrics = val_epoch(model, test_loader, criterion, cfg.device)
    print(f"Fold {fold} Test Result:")
    for k, v in test_metrics.items():
        print(f"  {k}: {v:.4f}")


[ADNI Dataset: ADCN] 样本分布：
  CN (0): 204
  AD (1): 219


==================== Fold 1 ====================
Train: 304, Val: 34, Test: 85


C:\Users\dongz\AppData\Local\Temp\ipykernel_49860\1132704963.py:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
C:\Users\dongz\AppData\Local\Temp\ipykernel_49860\1923002093.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
C:\Users\dongz\AppData\Local\Temp\ipykernel_49860\1923002093.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/20 | Train Loss: 1.9900 Acc: 0.5000 | Val Loss: 0.8502 Acc: 0.4706 AUC: 0.6267
  >>> New Best Model Saved (Acc: 0.4706)


C:\Users\dongz\AppData\Local\Temp\ipykernel_49860\1923002093.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
C:\Users\dongz\AppData\Local\Temp\ipykernel_49860\1923002093.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 2/20 | Train Loss: 1.5330 Acc: 0.4770 | Val Loss: 0.7003 Acc: 0.4706 AUC: 0.7101


...

C:\Users\dongz\AppData\Local\Temp\ipykernel_49860\1923002093.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
C:\Users\dongz\AppData\Local\Temp\ipykernel_49860\1923002093.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 20/20 | Train Loss: 0.7119 Acc: 0.4901 | Val Loss: 0.6968 Acc: 0.4706 AUC: 0.7292
Testing Fold 1...


C:\Users\dongz\AppData\Local\Temp\ipykernel_49860\1132704963.py:63: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path))
C:\Users

Fold 1 Test Result:
  ACC: 0.6824
  PRE: 0.7576
  SEN: 0.5682
  SPE: 0.8049
  F1: 0.6494
  AUC: 0.6871
  MCC: 0.3825


TypeError: unsupported format string passed to numpy.ndarray.__format__